# 09 — Point-in-Time Ranking Feature Engineering (MVP First)

**Input:** Notebook 08 `*_fusion_rrf_top500.parquet`, Notebook 02/03 DuckDB snapshots, all four RetailRocket CSVs through DuckDB views. **Output:** full-candidate ranking parquet + CSV/JSON diagnostics for Notebook 10 LightGBM.

**Leakage policy:** all features must be available at split cutoff `t`; future 7-day events are used **only** to create labels. Labels represent *observed future interactions*, not exposure-corrected negatives; unclicked/unobserved candidates are **weak negatives**. The 500-candidate pool creates severe imbalance. No sampled negatives are required for this MVP.

**Important:** per-split snapshots are point-in-time at the cutoff, *not* as-of each historical event. This is the same split-level contract as Notebook 08.

In [1]:
from pathlib import Path
import json, time, math
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

cwd=Path.cwd().resolve()
ROOT=next((p for p in (cwd,*cwd.parents) if (p/'data'/'processed'/'notebook02.duckdb').exists()),None)
assert ROOT is not None, 'Cannot find notebook02.duckdb; open VS Code in project root or notebooks folder.'
OUT=ROOT/'data'/'processed'; REPORT=ROOT/'data'/'reports'; REPORT.mkdir(parents=True,exist_ok=True)
RAW=ROOT/'data'/'raw'; DB=OUT/'notebook02.duckdb'
con=duckdb.connect(str(DB)); con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")
SPLITS=('train','validation','test'); DAY=86_400_000
CUT={s:int(con.execute(f'SELECT MIN(cutoff_ts) FROM {s}_examples').fetchone()[0]) for s in SPLITS}
def q(p): return "'"+str(p).replace(chr(92),'/').replace("'","''")+"'"
print('Root:',ROOT);print('Cutoffs:',{s:str(pd.to_datetime(t,unit='ms',utc=True)) for s,t in CUT.items()})
for s in SPLITS:
    path=OUT/f'{s}_fusion_rrf_top500.parquet'
    assert path.is_file(),f'Missing {path}: run Notebook 08 first'
    con.execute(f'CREATE OR REPLACE VIEW {s}_09_input AS SELECT * FROM read_parquet({q(path)})')


Root: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks
Cutoffs: {'train': '2015-07-31 19:47:53.596000+00:00', 'validation': '2015-08-18 18:21:27.439000+00:00', 'test': '2015-09-04 07:47:49.447000+00:00'}


## 1. Validate dependencies and input schema

The original four CSVs were incorporated into Notebook 02 DuckDB via `events_raw`, `props_all`, and `category_tree`. We reuse those materialized/virtual relations instead of rereading 20M property rows three times.

In [2]:
needed=['events_raw','props_all','category_tree','category_ancestry']
for s in SPLITS:
    needed += [f'{s}_positives',f'{s}_history',f'{s}_user_features',f'{s}_item_popularity',
               f'{s}_item_features',f'{s}_user_category',f'{s}_category_popularity']
existing={r[0] for r in con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='main'").fetchall()}
missing=[x for x in needed if x not in existing]
assert not missing,f'Missing DuckDB tables/views: {missing}. Run Notebooks 02–03.'
for s in SPLITS:
    cols=set(con.execute(f'DESCRIBE {s}_09_input').df()['column_name'])
    required={'user_id','item_id','retrieval_rank','retrieval_score','source_count','from_popularity',
              'from_cf','from_faiss','from_category','is_fallback'}
    assert required<=cols,(s,required-cols)
    n,u,dup=con.execute(f'''SELECT COUNT(*),COUNT(DISTINCT user_id),
      COUNT(*)-COUNT(DISTINCT (user_id,item_id)) FROM {s}_09_input''').fetchone()
    print(s,'rows',n,'users',u,'duplicates',dup)
    assert dup==0 and n==500*u


train rows 2055000 users 4110 duplicates 0
validation rows 1828500 users 3657 duplicates 0
test rows 1786000 users 3572 duplicates 0


## 2. Prepare pre-cutoff event statistics

One scan per split, **only** `ts < cutoff`. Compute 1/7/30-day counts by event type, user-item interactions, recency, item-level popularity. `events_raw` is sourced from `events.csv`. We intentionally use a fixed 30-day history for consistency with retrieval.

In [3]:
def build_event_features(s,t):
    con.execute(f'''CREATE OR REPLACE TABLE {s}_09_event30 AS
      SELECT user_id,item_id,ts,event FROM events_raw
      WHERE ts >= {t-30*DAY} AND ts < {t}''')
    con.execute(f'''CREATE OR REPLACE TABLE {s}_09_user AS
      SELECT user_id, COUNT(*)::BIGINT AS user_events_30d,
        COUNT(DISTINCT item_id)::BIGINT AS user_unique_items_30d,
        COUNT(*) FILTER (WHERE ts >= {t-7*DAY})::BIGINT AS user_events_7d,
        COUNT(*) FILTER (WHERE ts >= {t-DAY})::BIGINT AS user_events_1d,
        COUNT(*) FILTER (WHERE event='view')::BIGINT AS user_views_30d,
        COUNT(*) FILTER (WHERE event='addtocart')::BIGINT AS user_carts_30d,
        COUNT(*) FILTER (WHERE event='transaction')::BIGINT AS user_purchases_30d,
        COUNT(*) FILTER (WHERE event='view' AND ts >= {t-7*DAY})::BIGINT AS user_views_7d,
        COUNT(*) FILTER (WHERE event='addtocart' AND ts >= {t-7*DAY})::BIGINT AS user_carts_7d,
        COUNT(*) FILTER (WHERE event='transaction' AND ts >= {t-7*DAY})::BIGINT AS user_purchases_7d,
        ({t}-MAX(ts))::BIGINT AS user_last_event_age_ms
      FROM {s}_09_event30 GROUP BY user_id''')
    con.execute(f'''CREATE OR REPLACE TABLE {s}_09_item AS
      SELECT item_id, COUNT(*)::BIGINT AS item_events_30d,
        COUNT(DISTINCT user_id)::BIGINT AS item_users_30d,
        COUNT(*) FILTER (WHERE ts >= {t-7*DAY})::BIGINT AS item_events_7d,
        COUNT(*) FILTER (WHERE ts >= {t-DAY})::BIGINT AS item_events_1d,
        COUNT(*) FILTER (WHERE event='view')::BIGINT AS item_views_30d,
        COUNT(*) FILTER (WHERE event='addtocart')::BIGINT AS item_carts_30d,
        COUNT(*) FILTER (WHERE event='transaction')::BIGINT AS item_purchases_30d,
        COUNT(*) FILTER (WHERE event='view' AND ts >= {t-7*DAY})::BIGINT AS item_views_7d,
        COUNT(*) FILTER (WHERE event='transaction' AND ts >= {t-7*DAY})::BIGINT AS item_purchases_7d,
        ({t}-MAX(ts))::BIGINT AS item_last_event_age_ms
      FROM {s}_09_event30 GROUP BY item_id''')
    # Candidate users may have interacted with items before the 30d window;
    # here the interaction count is intentionally limited to 30d.
    con.execute(f'''CREATE OR REPLACE TABLE {s}_09_ui AS
      SELECT user_id,item_id,COUNT(*)::BIGINT AS ui_events_30d,
        COUNT(*) FILTER(WHERE event='view')::BIGINT AS ui_views_30d,
        ({t}-MAX(ts))::BIGINT AS ui_last_age_ms
      FROM {s}_09_event30 GROUP BY 1,2''')
    print(s,'event rows',con.execute(f'SELECT COUNT(*) FROM {s}_09_event30').fetchone()[0])
for s in SPLITS: build_event_features(s,CUT[s])


train event rows 677577
validation event rows 609565
test event rows 533842


## 3. Property, category and source-level feature views

Item property count/category come from **Notebook 02 split snapshot** built with `property_ts <= cutoff`. Property last-update timestamp is also computed with `ts < = cutoff` (the two `item_properties` CSVs feed `props_all`). We do **not** assume that a numeric `price` exists in RetailRocket; instead derive generic property count and category indicators. The static category hierarchy is sourced from `category_tree.csv`.

Retrieval source ranks/scores are sourced from the precomputed Notebook 04/05/07/08 candidate Parquets; missing source rank is encoded as 0 with an explicit `from_*` indicator.

In [4]:
SOURCE_FILES={'popularity':'pop_decayed_top100','cf':'cf_cooccurrence_top100',
              'faiss':'faiss_ivf_np16_top100','category':'fusion_category_top200'}
for s in SPLITS:
    for source,suffix in SOURCE_FILES.items():
        p=OUT/f'{s}_{suffix}.parquet'
        assert p.exists(),f'Missing source {p}'
        con.execute(f'''CREATE OR REPLACE VIEW {s}_09_src_{source} AS
          SELECT user_id::BIGINT AS user_id,item_id::BIGINT AS item_id,
                 retrieval_rank::INTEGER AS source_rank,
                 retrieval_score::DOUBLE AS source_score
          FROM read_parquet({q(p)})''')
    t=CUT[s]
    con.execute(f'''CREATE OR REPLACE TABLE {s}_09_prop_update AS
      SELECT p.item_id,MAX(p.ts)::BIGINT AS property_last_ts,
             COUNT(DISTINCT p.property)::BIGINT AS property_distinct_keys_historical_raw
      FROM props_all p
      JOIN (SELECT DISTINCT item_id FROM {s}_09_input) c ON p.item_id=c.item_id
      WHERE p.ts <= {t}
      GROUP BY p.item_id''')
    # Note: property_distinct_keys_historical_raw is a historical distinct-key count,
    # not a 30-day metric; keep explicit naming in output below.
    print(s,'property rows',con.execute(f'SELECT COUNT(*) FROM {s}_09_prop_update').fetchone()[0])


train property rows 57161
validation property rows 54570
test property rows 51740


## 4. Build candidate-level ranking dataset

**Label definition:** `label=1` only when the candidate pair is in `{split}_positives` (eligible 7-day future interactions); otherwise `0` means **not observed as a positive**, not a proven dislike. The feature SQL never references future interactions; the positives join is used only for the target column.

To prevent a subtle leakage route, `retrieval_rank` and source scores must have been computed entirely from pre-cutoff information by prior notebooks.

In [5]:
def build_ranking(s,t):
    source_joins='\n'.join(f'LEFT JOIN {s}_09_src_{name} src_{name} ON c.user_id=src_{name}.user_id AND c.item_id=src_{name}.item_id' for name in SOURCE_FILES)
    source_cols=',\n'.join(f'''COALESCE(src_{name}.source_rank,0)::INTEGER AS {name}_rank,
        COALESCE(src_{name}.source_score,0)::DOUBLE AS {name}_score''' for name in SOURCE_FILES)
    con.execute(f'''CREATE OR REPLACE TABLE {s}_09_ranking AS
      SELECT c.user_id::BIGINT AS user_id,c.item_id::BIGINT AS item_id,
        {t}::BIGINT AS cutoff_ts,
        CASE WHEN p.user_id IS NOT NULL THEN 1 ELSE 0 END::INTEGER AS label,
        c.retrieval_rank::INTEGER AS retrieval_rank,
        COALESCE(c.retrieval_score,0)::DOUBLE AS rrf_score,
        COALESCE(c.source_count,0)::INTEGER AS source_count,
        c.from_popularity::INTEGER AS from_popularity,
        c.from_cf::INTEGER AS from_cf,
        c.from_faiss::INTEGER AS from_faiss,
        c.from_category::INTEGER AS from_category,
        c.is_fallback::INTEGER AS is_fallback,
        {source_cols},
        COALESCE(u.user_events_30d,0)::BIGINT AS user_events_30d,
        COALESCE(u.user_unique_items_30d,0)::BIGINT AS user_unique_items_30d,
        COALESCE(u.user_events_7d,0)::BIGINT AS user_events_7d,
        COALESCE(u.user_events_1d,0)::BIGINT AS user_events_1d,
        COALESCE(u.user_views_30d,0)::BIGINT AS user_views_30d,
        COALESCE(u.user_carts_30d,0)::BIGINT AS user_carts_30d,
        COALESCE(u.user_purchases_30d,0)::BIGINT AS user_purchases_30d,
        COALESCE(u.user_views_7d,0)::BIGINT AS user_views_7d,
        COALESCE(u.user_carts_7d,0)::BIGINT AS user_carts_7d,
        COALESCE(u.user_purchases_7d,0)::BIGINT AS user_purchases_7d,
        COALESCE(u.user_last_event_age_ms,0)::BIGINT / {DAY}::DOUBLE AS user_recency_days,
        COALESCE(i.item_events_30d,0)::BIGINT AS item_events_30d,
        COALESCE(i.item_users_30d,0)::BIGINT AS item_users_30d,
        COALESCE(i.item_events_7d,0)::BIGINT AS item_events_7d,
        COALESCE(i.item_events_1d,0)::BIGINT AS item_events_1d,
        COALESCE(i.item_views_30d,0)::BIGINT AS item_views_30d,
        COALESCE(i.item_carts_30d,0)::BIGINT AS item_carts_30d,
        COALESCE(i.item_purchases_30d,0)::BIGINT AS item_purchases_30d,
        COALESCE(i.item_views_7d,0)::BIGINT AS item_views_7d,
        COALESCE(i.item_purchases_7d,0)::BIGINT AS item_purchases_7d,
        COALESCE(i.item_last_event_age_ms,0)::BIGINT / {DAY}::DOUBLE AS item_recency_days,
        COALESCE(ui.ui_events_30d,0)::BIGINT AS ui_events_30d,
        COALESCE(ui.ui_views_30d,0)::BIGINT AS ui_views_30d,
        COALESCE(ui.ui_last_age_ms,0)::BIGINT / {DAY}::DOUBLE AS ui_recency_days,
        COALESCE(f.known_property_count,0)::BIGINT AS known_property_count,
        COALESCE(f.category_id,-1)::BIGINT AS category_id,
        COALESCE(f.parent_category_id,-1)::BIGINT AS parent_category_id,
        COALESCE(a.root_category_id,-1)::BIGINT AS root_category_id,
        COALESCE(a.category_depth,-1)::INTEGER AS category_depth,
        CASE WHEN f.category_id IS NULL THEN 1 ELSE 0 END::INTEGER AS missing_category,
        CASE WHEN f.item_id IS NULL THEN 1 ELSE 0 END::INTEGER AS missing_properties,
        CASE WHEN pu.property_last_ts IS NULL THEN 1 ELSE 0 END::INTEGER AS missing_property_timestamp,
        COALESCE(({t}-pu.property_last_ts)/{DAY}::DOUBLE,0) AS days_since_property_update,
        COALESCE(pu.property_distinct_keys_historical_raw,0)::BIGINT AS property_distinct_keys_historical,
        COALESCE(uc.user_category_30d_events,0)::BIGINT AS user_category_30d_events,
        COALESCE(uc.user_category_30d_items,0)::BIGINT AS user_category_30d_items,
        COALESCE(cp.category_30d_events,0)::BIGINT AS category_30d_events,
        COALESCE(uc.user_category_30d_events,0)::DOUBLE /
            GREATEST(COALESCE(u.user_events_30d,0),1) AS user_category_affinity,
        COALESCE(i.item_carts_30d,0)::DOUBLE /
            GREATEST(COALESCE(i.item_views_30d,0),1) AS item_cart_to_view_ratio,
        COALESCE(i.item_purchases_30d,0)::DOUBLE /
            GREATEST(COALESCE(i.item_views_30d,0),1) AS item_purchase_to_view_ratio,
        COALESCE(u.user_purchases_30d,0)::DOUBLE /
            GREATEST(COALESCE(u.user_views_30d,0),1) AS user_purchase_to_view_ratio
      FROM {s}_09_input c
      LEFT JOIN {s}_positives p ON c.user_id=p.user_id AND c.item_id=p.item_id
      LEFT JOIN {s}_09_user u ON c.user_id=u.user_id
      LEFT JOIN {s}_09_item i ON c.item_id=i.item_id
      LEFT JOIN {s}_09_ui ui ON c.user_id=ui.user_id AND c.item_id=ui.item_id
      LEFT JOIN {s}_item_features f ON c.item_id=f.item_id
      LEFT JOIN category_ancestry a ON f.category_id=a.category_id
      LEFT JOIN {s}_user_category uc ON c.user_id=uc.user_id AND f.category_id=uc.category_id
      LEFT JOIN {s}_category_popularity cp ON f.category_id=cp.category_id
      LEFT JOIN {s}_09_prop_update pu ON c.item_id=pu.item_id
      {source_joins}''')
    return con.execute(f'''SELECT COUNT(*) AS rows,SUM(label) AS positives,
       COUNT(DISTINCT user_id) AS users,COUNT(DISTINCT item_id) AS items
       FROM {s}_09_ranking''').df()
for s in SPLITS:
    start=time.perf_counter(); stats=build_ranking(s,CUT[s]); print(s,'seconds',round(time.perf_counter()-start,2));display(stats)


train seconds 2.57


,rows,positives,users,items
0,2055000,2131.0,4110,60454


validation seconds 2.6


,rows,positives,users,items
0,1828500,1996.0,3657,57668


test seconds 2.36


,rows,positives,users,items
0,1786000,1785.0,3572,54978


## 5. Leakage and data-quality checks

We check timestamp bounds, 1:1 joins, target consistency, no seen items, and finite numeric features. **Important:** these checks are necessary but not a formal proof of leakage freedom; the prior retrieval stages must also be audited. In particular, we deliberately avoid Notebook 02 `*_examples` negatives because those sampled negatives are not the full retrieval candidate set.

In [6]:
CHECKS=[]
for s in SPLITS:
    t=CUT[s]; table=f'{s}_09_ranking'
    total,users,duplicates,positives=con.execute(f'''SELECT COUNT(*),COUNT(DISTINCT user_id),
      COUNT(*)-COUNT(DISTINCT (user_id,item_id)),SUM(label) FROM {table}''').fetchone()
    seen=con.execute(f'''SELECT COUNT(*) FROM {table} r JOIN {s}_history h
      ON r.user_id=h.user_id AND r.item_id=h.item_id''').fetchone()[0]
    label_bad=con.execute(f'''SELECT COUNT(*) FROM {table} r LEFT JOIN {s}_positives p
      ON r.user_id=p.user_id AND r.item_id=p.item_id
      WHERE r.label<>CASE WHEN p.user_id IS NOT NULL THEN 1 ELSE 0 END''').fetchone()[0]
    time_bad=con.execute(f'''SELECT COUNT(*) FROM {table}
      WHERE cutoff_ts<>{t} OR user_recency_days<0 OR item_recency_days<0
        OR days_since_property_update<0 OR user_category_affinity<0
        OR user_category_affinity>1.000001''').fetchone()[0]
    future_event=con.execute(f'''SELECT COUNT(*) FROM {s}_09_event30 WHERE ts>={t}''').fetchone()[0]
    future_props=con.execute(f'''SELECT COUNT(*) FROM {s}_09_prop_update WHERE property_last_ts>{t}''').fetchone()[0]
    invalid_rank=con.execute(f'''SELECT COUNT(*) FROM (SELECT user_id,COUNT(*) n,MIN(retrieval_rank) lo,
      MAX(retrieval_rank) hi FROM {table} GROUP BY user_id)
      WHERE n<>500 OR lo<>1 OR hi<>500''').fetchone()[0]
    row=dict(split=s,rows=int(total),users=int(users),positives=int(positives),
      positive_rate=float(positives/total),duplicate_pairs=int(duplicates),
      seen_item_leakage=int(seen),label_mismatch=int(label_bad),time_feature_errors=int(time_bad),
      future_events_in_features=int(future_event),future_properties_in_features=int(future_props),
      invalid_user_rank_groups=int(invalid_rank))
    CHECKS.append(row)
    assert not any(row[k] for k in ['duplicate_pairs','seen_item_leakage','label_mismatch',
      'time_feature_errors','future_events_in_features','future_properties_in_features','invalid_user_rank_groups']),row
checks_df=pd.DataFrame(CHECKS);display(checks_df)


,split,rows,users,positives,positive_rate,duplicate_pairs,seen_item_leakage,label_mismatch,time_feature_errors,future_events_in_features,future_properties_in_features,invalid_user_rank_groups
0,train,2055000,4110,2131,0.001037,0,0,0,0,0,0,0
1,validation,1828500,3657,1996,0.001092,0,0,0,0,0,0,0
2,test,1786000,3572,1785,0.000999,0,0,0,0,0,0,0


## 6. Missing-value policy, LightGBM feature manifest, distributions

Missing source ranks and scores are 0 (absent source); missing category IDs are -1; missing numeric counts are 0; missing recencies are 0 **paired with missingness flags when relevant**. `user_id`, `item_id`, `cutoff_ts`, and `label` are **not** model features. `retrieval_rank` is useful as a baseline but may cause the model to copy RRF ranking; Notebook 10 should compare models with and without it. Category IDs are integer codes and should be declared categorical for LightGBM, rather than treated as ordered magnitudes.

In [7]:
EXCLUDE={'user_id','item_id','cutoff_ts','label'}
FEATURES=[c for c in con.execute('DESCRIBE train_09_ranking').df().column_name if c not in EXCLUDE]
CATEGORICAL=['category_id','parent_category_id','root_category_id']
assert all(c in FEATURES for c in CATEGORICAL)
FEATURE_GROUPS={
 'retrieval':[x for x in FEATURES if x in {'retrieval_rank','rrf_score','source_count','is_fallback'} or x.startswith(('from_','popularity_rank','popularity_score','cf_rank','cf_score','faiss_rank','faiss_score','category_rank','category_score'))],
 'user':[x for x in FEATURES if x.startswith('user_') and not x.startswith('user_category_')],
 'item':[x for x in FEATURES if x.startswith('item_')],
 'user_item':[x for x in FEATURES if x.startswith('ui_')],
 'category':[x for x in FEATURES if 'category' in x or x in {'parent_category_id','root_category_id'}],
 'property':[x for x in FEATURES if 'property' in x or 'properties' in x],
 'temporal':[x for x in FEATURES if 'recency' in x or 'days_since' in x],
}
print('Feature count:',len(FEATURES)); print('Groups:',{k:len(v) for k,v in FEATURE_GROUPS.items()})
print('Categorical features:',CATEGORICAL)


Feature count: 57
Groups: {'retrieval': 16, 'user': 12, 'item': 12, 'user_item': 3, 'category': 12, 'property': 5, 'temporal': 4}
Categorical features: ['category_id', 'parent_category_id', 'root_category_id']


In [8]:
DISTRIBUTIONS=[]
for s in SPLITS:
    t=f'{s}_09_ranking'
    # SQL-side aggregation avoids loading ~2 million candidate rows into pandas.
    for feat in FEATURES:
        col='"'+feat+'"'
        count,nulls,minimum,maximum,avg,std=con.execute(f'''SELECT COUNT(*),
          COUNT(*) FILTER(WHERE {col} IS NULL OR NOT isfinite({col}::DOUBLE)),
          MIN({col}::DOUBLE),MAX({col}::DOUBLE),AVG({col}::DOUBLE),STDDEV_POP({col}::DOUBLE)
          FROM {t}''').fetchone()
        DISTRIBUTIONS.append(dict(split=s,feature=feat,rows=int(count),missing_or_nonfinite=int(nulls),
          min=minimum,max=maximum,mean=avg,std=std))
dist_df=pd.DataFrame(DISTRIBUTIONS)
assert dist_df.missing_or_nonfinite.sum()==0,'Nonfinite/missing numeric features detected'
display(dist_df[dist_df.split=='validation'].head(25))


,split,feature,rows,missing_or_nonfinite,min,max,mean,std
57,validation,retrieval_rank,1828500,0,1.0,500.000000,250.500000,144.337279
58,validation,rrf_score,1828500,0,0.0,0.050348,0.005095,0.004947
59,validation,source_count,1828500,0,0.0,4.000000,0.785526,0.576216
60,validation,from_popularity,1828500,0,0.0,1.000000,0.200000,0.400000
61,validation,from_cf,1828500,0,0.0,1.000000,0.067782,0.251372
62,validation,from_faiss,1828500,0,0.0,1.000000,0.158053,0.364791
63,validation,from_category,1828500,0,0.0,1.000000,0.359690,0.479910
64,validation,is_fallback,1828500,0,0.0,1.000000,0.288329,0.452985
65,validation,popularity_rank,1828500,0,0.0,100.000000,10.100000,23.972693
66,validation,popularity_score,1828500,0,0.0,858.257033,17.208946,51.944905


## 7. Save outputs for Notebook 10

Parquet is the primary model input. CSV files are compact diagnostics, not millions of candidate rows. For a production-scale implementation, replace full candidate table materialization with streamed partitioned Parquet.

In [9]:
EXPORTS=[]
for s in SPLITS:
    target=OUT/f'{s}_ranking_features_top500.parquet'
    con.execute(f'''COPY (SELECT * FROM {s}_09_ranking ORDER BY user_id,retrieval_rank)
      TO {q(target)} (FORMAT PARQUET, COMPRESSION ZSTD)''')
    assert target.exists() and target.stat().st_size>0
    EXPORTS.append(dict(split=s,path=target.name,size_mb=round(target.stat().st_size/1048576,2)))
checks_df.to_csv(REPORT/'notebook09_quality_checks.csv',index=False)
dist_df.to_csv(REPORT/'notebook09_feature_distributions.csv',index=False)
manifest={'feature_columns':FEATURES,'categorical_columns':CATEGORICAL,'excluded_columns':sorted(EXCLUDE),
          'feature_groups':FEATURE_GROUPS,'missing_policy':{'counts':'0','ranks_and_scores':'0 for absent source',
          'category_ids':'-1','recency':'0; check related missingness indicators'},
          'label_policy':'eligible future positive=1, all other retrieved candidates=0 (weak negatives)',
          'split_cutoffs_ms':CUT}
(REPORT/'notebook09_feature_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
report={'notebook':'09_feature_engineering','inputs':'Notebook 08 fused top500; Notebook 02/03 split PIT tables; four original CSV lineage',
        'point_in_time_rule':'all features as of split cutoff; future 7d interactions used only as labels',
        'event_level_pit':False,'source_files':['events.csv','item_properties_part1.csv',
          'item_properties_part2.csv','category_tree.csv'],
        'checks':CHECKS,'feature_count':len(FEATURES),'feature_groups':FEATURE_GROUPS,
        'categorical_features':CATEGORICAL,'exports':EXPORTS,
        'limitations':['All non-positive candidate pairs are weak negatives; no impression logs',
          'No per-event historical category join; category snapshot is at split cutoff',
          'Feature rank/score provenance depends on pre-cutoff notebooks 04–08',
          'Do not tune using test split; train on train and select on validation',
          'Future positives denominator includes only eligible positives, not all future events']}
(REPORT/'notebook09_feature_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Exports:');display(pd.DataFrame(EXPORTS))
print('Report:',REPORT/'notebook09_feature_report.json')


Exports:


,split,path,size_mb
0,train,train_ranking_features_top500.parquet,37.03
1,validation,validation_ranking_features_top500.parquet,32.56
2,test,test_ranking_features_top500.parquet,31.94


Report: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook09_feature_report.json


## 8. Handoff to Notebook 10 — LightGBM Pre-Ranking

- Train: `train_ranking_features_top500.parquet`; validate: `validation_ranking_features_top500.parquet`; test: `test_ranking_features_top500.parquet`.
- Read `notebook09_feature_manifest.json` for features and categorical columns. Keep groups by `user_id` when training ranking objectives.
- Baselines: RRF Top-20/50/100; LightGBM binary classifier and/or `lambdarank` with group sizes, compare Recall@K / NDCG@K / HitRate@K.
- Use **validation only** for tuning and model selection. Test is confirmation.
- Since positives are sparse and weak negatives are numerous, track positive rate and consider negative downsampling **only in the training fold**, with evaluation on full 500-candidate lists.
- No train/test feature fitting, global target encoding, or future property data.